# Fast Chow Polynomial Computation for $K_n$

This implementation calculates the Chow polynomial $H_{K_n}(y)$ using the **Integer Partition Recurrence**.

## 1. The Formula

$$H_{K_{n+1}}(y) = (1 - ny) H_{K_n}(y) + y \sum_{\lambda \vdash n} C_\lambda \cdot |\lambda| \cdot H_{K_{|\lambda|}}(y) \cdot H_{\oplus}(\lambda, y)$$

Where:
* $\lambda \vdash n$: An integer partition of $n$.
* $|\lambda|$: The number of parts in $\lambda$.
* $C_\lambda$: The number of set partitions of $\{1,\dots,n\}$ with block sizes $\lambda$.
* $H_{\oplus}(\lambda, y)$: The Chow polynomial of the direct sum $K_{\lambda_1} \oplus K_{\lambda_2} \oplus \dots$

## 2. Optimization Strategy

To make this computationally feasible for $N \ge 9$:
1. **Direct Sum via Matroid:** $H_{\oplus}(\lambda)$ is computed by building the direct sum matroid $K_{\lambda_1} \oplus \cdots \oplus K_{\lambda_k}$ and calling `get_chow_polynomial` (Chow does *not* factor as a simple product over direct sums).
2. **Shape Caching:** Cache invariants by sorted tuple of clique sizes in `chowBraid.json`.

In [10]:
import sys
import os

current_dir = os.path.abspath(os.getcwd())
if os.path.exists(os.path.join(current_dir, "conclusion", "explorations on complete graphs")):
    project_root = current_dir
else:
    project_root = os.path.dirname(os.path.dirname(current_dir))
sys.path.insert(0, project_root)
sys.path.insert(0, os.path.join(project_root, "the-machine"))

from sage.all import *
from HM_module import *
from custom_matroid_functions import *

In [11]:
import json
from collections import Counter

P = LaurentPolynomialRing(QQ, 'x')
x = P.gen()

# --- Cache paths (chowBraid.json) ---
_expl_dir = os.path.join(os.getcwd(), "conclusion", "explorations on complete graphs")
CHOWBRAID_CACHE_PATH = os.path.join(_expl_dir, "chowBraid.json") if os.path.exists(_expl_dir) else os.path.join(os.getcwd(), "chowBraid.json")

# --- Global Caches ---
_Kn_cache = {}
_invariant_cache = {}

def _parse_poly(s):
    if s == "1" or s == 1:
        return P(1)
    return P(str(s))

def _load_chowBraid_cache():
    """Load _Kn_cache and _invariant_cache from chowBraid.json."""
    if os.path.exists(CHOWBRAID_CACHE_PATH):
        try:
            with open(CHOWBRAID_CACHE_PATH, 'r') as f:
                data = json.load(f)
            kn = {int(k): _parse_poly(v) for k, v in data.get("H_Kn", {}).items()}
            inv = {tuple(json.loads(k)): _parse_poly(v) for k, v in data.get("H_direct_sum", {}).items()}
            return kn, inv
        except (json.JSONDecodeError, ValueError) as e:
            print(f"Warning: Could not load chowBraid cache: {e}")
    return {}, {}

def _save_chowBraid_cache():
    """Save _Kn_cache and _invariant_cache to chowBraid.json."""
    try:
        h_kn = {str(int(k)): str(v) for k, v in _Kn_cache.items()}
        h_ds = {json.dumps([int(x) for x in k]): str(v) for k, v in _invariant_cache.items()}
        with open(CHOWBRAID_CACHE_PATH, 'w') as f:
            json.dump({"H_Kn": h_kn, "H_direct_sum": h_ds}, f, indent=2)
    except Exception as e:
        print(f"Warning: Could not save chowBraid cache: {e}")

def C_lambda(lam, n):
    """Number of set partitions of {1..n} with block sizes lam."""
    mult = Counter(lam)
    denom = 1
    for j, mj in mult.items():
        denom *= factorial(int(j)) ** int(mj) * factorial(int(mj))
    return factorial(int(n)) // denom

def matroid_direct_sum_cliques(lam):
    """Build matroid K_{a1} ⊕ K_{a2} ⊕ ... for lam = [a1, a2, ...]."""
    if not lam:
        return Matroid(Matrix(QQ, 0, 0))
    G = Graph()
    offset = 0
    for a in lam:
        if a >= 2:
            for i in range(a):
                for j in range(i + 1, a):
                    G.add_edge(offset + i, offset + j)
        else:
            G.add_vertex(offset)
        offset += max(a, 1)
    return Matroid(G)

def get_invariant(shape):
    """Chow polynomial for direct sum of cliques via get_chow_polynomial (matroid-based)."""
    shape = tuple(sorted(int(x) for x in shape))
    if shape in _invariant_cache:
        return _invariant_cache[shape]
    M = matroid_direct_sum_cliques(list(shape))
    result = P(get_chow_polynomial(M))
    _invariant_cache[shape] = result
    return result

def compute_sequence(N, use_cache=True):
    """Compute H_K1 through H_KN using the integer partition recurrence.
    Set use_cache=False to ignore chowBraid.json and recompute from scratch (e.g. if cache was corrupted)."""
    global _Kn_cache, _invariant_cache
    if use_cache:
        _Kn_cache, _invariant_cache = _load_chowBraid_cache()
    else:
        _Kn_cache, _invariant_cache = {}, {}
    _Kn_cache[1] = P(1)
    _invariant_cache[(1,)] = P(1)
    print("H_K1: 1")
    for n in range(1, N):
        term_base = (1 - n*x) * _Kn_cache[n]
        term_sum = P(0)
        for lam in Partitions(n):
            lam_list = list(lam)
            k = len(lam_list)
            C = C_lambda(lam_list, n)
            H_cont = _Kn_cache[k]
            H_restr = get_invariant(tuple(int(x) for x in lam_list))
            term_sum += x * C * k * H_cont * H_restr
        H_next = term_base + term_sum
        _Kn_cache[n+1] = H_next
        _invariant_cache[(n+1,)] = H_next
        print(f"H_K{n+1} computed. (Degree: {H_next})")
    _save_chowBraid_cache()
    return _Kn_cache[N]

In [12]:
N_TARGET = 9
final_poly = compute_sequence(N_TARGET, use_cache=False)  # use_cache=False to avoid corrupted cache
print(f"\nH_K{N_TARGET}:", final_poly)

H_K1: 1
H_K2 computed. (Degree: 1)
H_K3 computed. (Degree: 1 + x)
H_K4 computed. (Degree: 1 + 8*x + x^2)
H_K5 computed. (Degree: 1 + 41*x + 41*x^2 + x^3)
H_K6 computed. (Degree: 1 + 187*x + 732*x^2 + 187*x^3 + x^4)
H_K7 computed. (Degree: 1 + 855*x + 9556*x^2 + 9556*x^3 + 855*x^4 + x^5)
H_K8 computed. (Degree: 1 + 4111*x + 111211*x^2 + 299882*x^3 + 111211*x^4 + 4111*x^5 + x^6)
H_K9 computed. (Degree: 1 + 21110*x + 1260636*x^2 + 7507485*x^3 + 7507485*x^4 + 1260636*x^5 + 21110*x^6 + x^7)

H_K9: 1 + 21110*x + 1260636*x^2 + 7507485*x^3 + 7507485*x^4 + 1260636*x^5 + 21110*x^6 + x^7


In [ ]:
# Validate series method (fast_RHS_series.py)
import sys, os
_expl = os.path.join(os.getcwd(), "conclusion", "explorations on complete graphs")
if os.path.exists(_expl):
    sys.path.insert(0, _expl)
else:
    sys.path.insert(0, os.getcwd())  # notebook run from explorations folder
from fast_RHS_series import fast_RHS_series_method, validate_series

n_val = 6
H_K_ser, D_ser = fast_RHS_series_method(n_val, P, x, _Kn_cache)
ok, result = validate_series(n_val, P, x, _Kn_cache)
print(f"Series validation (n={n_val}): {'OK' if ok else 'FAILED'}")
print("H_K_series (trunc):", H_K_ser.truncate(n_val+2))
print("D_series (trunc):", D_ser.truncate(4))